# HealthBot

HealthBot is an AI-powered health education assistant that retrieves
medical information from web sources, generates a patient-friendly
summary, creates a comprehension quiz, and evaluates the user's answer.


## Imports & env Configuration

In [ ]:
# Standard Library
import os
import re
import time
from typing import Any, TypedDict

# Environment Variables
from dotenv import load_dotenv

# LangChain / LangGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END

# Health Topic Validation
from rapidfuzz import fuzz, process

## Environment Configuration & Validation

In [ ]:
# Load environment variables
load_dotenv("config.env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Validate required API credentials
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing. Check your config.env file."
    )

if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY is missing. Check your config.env file."
    )

print("Required API credentials loaded successfully.")

## API Configurations

This section configures the Gemini language model and Tavily search tool that will be used by the HealthBot LangGraph workflow.

In [ ]:
# Gemini Model Configuration
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

# Tavily Model Configuration
tavily_tool = TavilySearch(
    max_results=5,
    tavily_api_key=TAVILY_API_KEY
)

In [ ]:
# # Testing external api 

# # Test Gemini
# try:
#     response = gemini_llm.invoke("Reply with: Gemini API working")
#     print("Gemini:", response.content)

# except Exception as e:
#     print("Gemini API failed:", e)

# # Test Tavily
# try:
#     response = tavily_tool.invoke({"query": "diabetes medical information"})
#     print("Tavily:", "API working" if response.get("results") else "No results")

# except Exception as e:
#     print("Tavily API failed:", e)

## Helper Functions

In [ ]:
# General utility functions

# Return True if value is a non-empty string.
def is_valid_text(value: Any) -> bool:
    return isinstance(value, str) and bool(value.strip())

# Retry an operation after temporary failures.
def retry_operation(operation, max_attempts=3, delay=2):
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            return operation()
        except Exception as e:
            last_error = e
            print(
                f"Attempt {attempt}/{max_attempts} failed: {e}"
            )
            if attempt < max_attempts:
                time.sleep(delay)
    raise RuntimeError(
        f"Operation failed after {max_attempts} attempts: {last_error}"
    )

# Safely format a prompt using the supplied variables.
def format_prompt(template: str, **kwargs) -> str:
    if not is_valid_text(template):
        raise ValueError("Prompt template cannot be empty.")
    try:
        return template.format(**kwargs)
    except KeyError as e:
        raise ValueError(
            f"Missing prompt variable: {e.args[0]}"
        ) from e
    except Exception as e:
        raise ValueError(
            f"Prompt formatting failed: {e}"
        ) from e

In [ ]:
# AI and external service helper functions

# Retrieve and validate medical information from Tavily.
def search_medical_information(topic: str) -> list[dict[str, Any]]:
    if not is_valid_text(topic):
        raise ValueError("Health topic cannot be empty.")
    
    def perform_search():
        response = tavily_tool.invoke(
            {
                "query": (
                    f"{topic} medical information "
                    "reputable sources"
                )
            }
        )
        if not isinstance(response, dict):
            raise ValueError(
                "Tavily returned an unexpected response format."
            )
        results = response.get("results", [])
        if not isinstance(results, list):
            raise ValueError(
                "Tavily results are in an unexpected format."
            )
        valid_results = []
        for result in results:
            if not isinstance(result, dict):
                continue
            content = result.get("content", "")
            if not is_valid_text(content):
                continue
            valid_results.append(
                {
                    "title": result.get("title", ""),
                    "url": result.get("url", ""),
                    "content": content,
                    "score": result.get("score", 0)
                }
            )
        if not valid_results:
            raise ValueError(
                "Tavily returned no usable medical information."
            )
        return valid_results

    return retry_operation(perform_search)

# Generate and validate text output from Gemini.
def generate_with_gemini(prompt: str):
    if not is_valid_text(prompt):
        raise ValueError("Prompt cannot be empty.")

    def generate():
        response = gemini_llm.invoke(prompt)
        content = getattr(response, "content", None)
        if content is None:
            raise ValueError(
                "Gemini returned no content."
            )

        if isinstance(content, str):
            text = content.strip()

        elif isinstance(content, list):
            text_parts = []
            for item in content:
                if isinstance(item, str):
                    text_parts.append(item)
                elif isinstance(item, dict):
                    if item.get("type") == "text":
                        value = item.get("text", "")

                        if value:
                            text_parts.append(value)
            text = "\n".join(text_parts).strip()
        else:
            text = str(content).strip()
        if not text:
            raise ValueError(
                "Gemini returned an empty response."
            )
        return text

    return retry_operation(generate)

# Extract a valid A-D grade from Gemini's response.
VALID_GRADES = {"A", "B", "C", "D"}
def extract_grade(feedback: str) -> str:
    if not is_valid_text(feedback):
        return ""

    match = re.search(
        r"\bGrade\s*[:\-]?\s*([ABCD])\b",
        feedback,
        re.IGNORECASE
    )

    if match:
        grade = match.group(1).upper()

        if grade in VALID_GRADES:
            return grade

    return ""

## LangGraph State Definition

In [ ]:
# Stores the shared state passed between HealthBot workflow nodes.
class HealthBotState(TypedDict, total=False):
    topic: str
    search_results: list[dict[str, Any]]
    summary: str
    quiz_question: str
    user_answer: str
    grade: str
    feedback: str
    ready_for_quiz: bool
    quiz_completed: bool
    continue_session: bool
    error: str

In [ ]:
# Create a consistent error state
def create_error_state(message: str) -> HealthBotState:
    return {"error": message}

In [ ]:
# Reset the workflow state for a new session
def reset_state() -> HealthBotState:
    return {
        "topic": "",
        "search_results": [],
        "summary": "",
        "quiz_question": "",
        "user_answer": "",
        "grade": "",
        "feedback": "",
        "ready_for_quiz": False,
        "quiz_completed": False,
        "continue_session": False,
        "error": ""
    }

## Prompt templates

HealthBot uses four dedicated prompts for:

1. Medical information summarization
2. Comprehension quiz generation
3. Patient answer grading
4. Topic classification 

Each prompt has a clearly defined source of information to reduce unsupported or hallucinated content.

In [ ]:
# Prompt templates used by the HealthBot workflow

SUMMARY_PROMPT = """
You are HealthBot, an AI-powered patient education assistant.

Create a clear, patient-friendly explanation of the health topic using ONLY
the medical information provided in the search results.

HEALTH TOPIC:
{topic}

MEDICAL SEARCH RESULTS:
{search_results}

Instructions:
1. Use only information from the medical search results.
2. Do not use outside knowledge or invent medical facts.
3. Write exactly 3 to 4 clear paragraphs.
4. Use simple, patient-friendly language.
5. Include important information needed to understand the topic.
6. Do not diagnose the patient.
7. Do not provide personalized medical advice.
8. Do not prescribe medication or recommend changing treatment.
9. If the search results do not contain enough reliable information, clearly
   state that there is insufficient information.
10. Preserve relevant source information for later reference.

Keep the response educational, neutral, clear, and easy to understand.
"""


QUIZ_PROMPT = """
Create exactly ONE comprehension question based ONLY on the
patient-friendly summary below.

SUMMARY:
{summary}

Instructions:
1. Use only information contained in the summary.
2. Do not use outside knowledge.
3. Do not introduce facts that are not present in the summary.
4. Make the question answerable using the summary alone.
5. Test an important concept from the summary.
6. Do not ask for a diagnosis or personalized medical advice.
7. Return only the question.

Question:
"""


GRADING_PROMPT = """
Evaluate the patient's answer to the comprehension question using ONLY
the provided summary as the source of truth.

TOPIC:
{topic}

SUMMARY:
{summary}

QUESTION:
{quiz_question}

PATIENT ANSWER:
{user_answer}

Instructions:
1. Evaluate the answer using only information in the summary.
2. Do not use outside knowledge or introduce unsupported medical facts.
3. Determine whether the answer demonstrates understanding of the question.
4. Assign one grade:
   - A = Correct and complete
   - B = Mostly correct
   - C = Partially correct
   - D = Incorrect, unsupported, or does not answer the question
5. Explain clearly why the grade was assigned.
6. Identify relevant evidence from the summary.
7. Include the relevant source or citation when available.
8. Keep the feedback concise and patient-friendly.
9. Do not diagnose the patient or provide personalized medical advice.

Return the result using exactly this structure:

Grade: <A/B/C/D>

Explanation:
<Why the grade was assigned>

Evidence:
<Relevant information from the summary>

Source:
<Relevant source or citation>
"""


TOPIC_CLASSIFICATION_PROMPT = """
You are a health-topic classifier for HealthBot.

Determine whether the user's topic is related to health, medicine,
healthcare, disease, symptoms, nutrition, fitness, physical wellbeing,
mental wellbeing, medical treatment, or prevention.

User topic:
{topic}

Return exactly two lines:

Classification: HEALTH or NON_HEALTH
Corrected Topic: <correctly spelled and standardized topic>

Rules:
- If the topic is a health-related topic, return HEALTH.
- Correct obvious spelling mistakes in the topic.
- Preserve the user's intended health topic.
- Do not invent a different topic.
- If the topic is not health-related, return NON_HEALTH.
- For NON_HEALTH, use "none" as the corrected topic.
"""

## Health Topic Vocabulary

In [ ]:
# Health vocabulary used for topic validation

DISEASES_AND_CONDITIONS = {
    "diabetes",
    "type 1 diabetes",
    "type 2 diabetes",
    "prediabetes",
    "hypertension",
    "high blood pressure",
    "low blood pressure",
    "high cholesterol",
    "heart disease",
    "coronary artery disease",
    "heart attack",
    "stroke",
    "asthma",
    "copd",
    "pneumonia",
    "bronchitis",
    "tuberculosis",
    "flu",
    "influenza",
    "covid",
    "covid-19",
    "common cold",
    "cancer",
    "breast cancer",
    "lung cancer",
    "prostate cancer",
    "skin cancer",
    "arthritis",
    "osteoarthritis",
    "rheumatoid arthritis",
    "osteoporosis",
    "migraine",
    "epilepsy",
    "anemia",
    "thyroid disease",
    "hypothyroidism",
    "hyperthyroidism",
    "pcos",
    "endometriosis",
    "ibs",
    "gerd",
    "kidney disease",
    "liver disease",
    "fatty liver",
    "hepatitis",
    "ulcer",
    "gastritis",
    "obesity",
    "depression",
    "anxiety",
    "insomnia",
    "dementia",
    "alzheimer's disease",
    "parkinson's disease"
}


SYMPTOMS = {
    "fever",
    "cough",
    "headache",
    "migraine",
    "fatigue",
    "weakness",
    "dizziness",
    "nausea",
    "vomiting",
    "diarrhea",
    "constipation",
    "abdominal pain",
    "chest pain",
    "back pain",
    "joint pain",
    "muscle pain",
    "sore throat",
    "runny nose",
    "shortness of breath",
    "difficulty breathing",
    "rapid heartbeat",
    "palpitations",
    "swelling",
    "rash",
    "itching",
    "bleeding",
    "weight loss",
    "weight gain",
    "loss of appetite",
    "insomnia"
}


BODY_SYSTEMS = {
    "heart",
    "brain",
    "lungs",
    "kidneys",
    "liver",
    "stomach",
    "intestines",
    "pancreas",
    "thyroid",
    "skin",
    "bones",
    "muscles",
    "blood",
    "immune system",
    "nervous system",
    "digestive system",
    "respiratory system",
    "cardiovascular system"
}


HEALTH_AND_WELLNESS = {
    "nutrition",
    "healthy diet",
    "diet",
    "exercise",
    "physical activity",
    "sleep",
    "sleep health",
    "mental health",
    "stress",
    "stress management",
    "weight management",
    "hydration",
    "vaccination",
    "vaccines",
    "immunization",
    "first aid",
    "hygiene",
    "preventive care",
    "health screening",
    "sexual health",
    "maternal health",
    "child health",
    "elderly health"
}


MEDICAL_CONCEPTS = {
    "medication",
    "medicine",
    "drug",
    "treatment",
    "therapy",
    "side effects",
    "dosage",
    "surgery",
    "physical therapy",
    "chemotherapy",
    "radiation therapy",
    "immunotherapy",
    "diagnosis",
    "prevention",
    "symptoms",
    "causes",
    "risk factors"
}


# Combine all categories into one validation vocabulary
HEALTH_TOPICS = (
    DISEASES_AND_CONDITIONS
    | SYMPTOMS
    | BODY_SYSTEMS
    | HEALTH_AND_WELLNESS
    | MEDICAL_CONCEPTS
)

## Health Topic Validation

In [ ]:
# Normalize user input for topic comparison
def normalize_topic(topic: str) -> str:
    if not topic:
        return ""
    topic = topic.lower().strip()
    topic = re.sub(r"[-_/]", " ", topic)
    topic = re.sub(r"\s+", " ", topic)
    return topic

# Check for an exact match in the local health vocabulary
def exact_health_match(topic: str) -> bool:
    normalized_topic = normalize_topic(topic)
    return normalized_topic in HEALTH_TOPICS

# Find a close match for spelling mistakes and minor variations
FUZZY_THRESHOLD = 85
def fuzzy_health_match(topic: str):
    normalized_topic = normalize_topic(topic)
    if not normalized_topic:
        return None, 0
    result = process.extractOne(
        normalized_topic,
        list(HEALTH_TOPICS),
        scorer=fuzz.ratio
    )
    if result is None:
        return None, 0
    matched_topic, score, _ = result
    if score >= FUZZY_THRESHOLD:
        return matched_topic, score
    return None, score

# Use Gemini as the final validation and spelling-correction fallback
def classify_with_gemini(topic: str):
    if not is_valid_text(topic):
        return False, ""
    prompt = format_prompt(
        TOPIC_CLASSIFICATION_PROMPT,
        topic=topic
    )
    try:
        response = generate_with_gemini(prompt)
        classification_match = re.search(
            r"Classification:\s*(HEALTH|NON_HEALTH)",
            response,
            re.IGNORECASE
        )
        corrected_match = re.search(
            r"Corrected Topic:\s*(.+)",
            response,
            re.IGNORECASE
        )
        if not classification_match:
            print("Gemini returned an invalid classification.")
            return False, ""
        classification = (
            classification_match.group(1).upper()
        )
        corrected_topic = ""
        if corrected_match:
            corrected_topic = normalize_topic(
                corrected_match.group(1)
            )
        if classification == "HEALTH":
            return True, corrected_topic
        return False, ""
    except Exception as e:
        print(f"Health classification failed: {e}")
        return False, ""

In [ ]:
# Complete health-topic validation pipeline
def validate_health_topic(topic: str) -> tuple[bool, str]:
    """
    Validate and normalize a user-provided health topic.

    Validation order:
    1. Exact match
    2. Fuzzy match
    3. Gemini classification and spelling correction
    """

    if not is_valid_text(topic):
        return False, ""

    normalized_topic = normalize_topic(topic)

    # 1. Exact match
    if exact_health_match(normalized_topic):
        print("Validation method: Exact match")
        return True, normalized_topic

    # 2. Fuzzy match
    fuzzy_match, fuzzy_score = fuzzy_health_match(
        normalized_topic
    )

    if fuzzy_match is not None:
        print("Validation method: Fuzzy match")
        print(f"Matched topic: {fuzzy_match}")
        print(f"Fuzzy score: {fuzzy_score:.1f}")
        return True, fuzzy_match

    # 3. Gemini fallback
    print("Using Gemini classifier...")

    is_health_topic, corrected_topic = (
        classify_with_gemini(normalized_topic)
    )

    if is_health_topic:
        return True, corrected_topic

    return False, ""

## LangGraph Workflow Nodes

This section implements the individual LangGraph nodes that perform the HealthBot workflow.

Each node has a single responsibility and communicates with other nodes through the shared `HealthBotState`.

### - Get Topic Node

In [ ]:
# User interaction functions
def get_topic(state: HealthBotState) -> HealthBotState:
    while True:
        topic = input(
            "\nWhat health topic or medical condition "
            "would you like to learn about?\n> "
        ).strip()

        # Reject empty input
        if not topic:
            print("Please enter a health topic.")
            continue

        # Validate and normalize the health topic
        is_valid, corrected_topic = validate_health_topic(topic)

        if not is_valid:
            print(
                "\nThe topic does not appear to be "
                "health-related."
            )
            print("Please enter a medical or health topic.")
            continue

        # Use the corrected topic when available
        final_topic = corrected_topic or topic
        if final_topic.lower() != topic.lower():
            print(
                f"\nCorrected health topic: {final_topic}"
            )
        else:
            print(
                f"\nHealth topic accepted: {final_topic}"
            )

        return {
            "topic": final_topic,
            "error": ""
        }

### - Search Node

In [ ]:
# Search for medical information based on the validated topic
def search_node(state: HealthBotState) -> HealthBotState:
    topic = state.get("topic", "")
    if not is_valid_text(topic):
        return {
            "search_results": [],
            "error": "Health topic is missing."
        }

    try:
        results = search_medical_information(topic)
        if not results:
            return {
                "search_results": [],
                "error": "No medical information found."
            }
        return {
            "search_results": results,
            "error": ""
        }
    except Exception as e:
        return {
            "search_results": [],
            "error": f"Medical search failed: {e}"
        }

### - Summarization Node

In [ ]:
# Generate a patient-friendly summary from medical search results
def summarize_information(state: HealthBotState) -> HealthBotState:
    topic = state.get("topic", "")
    search_results = state.get("search_results", [])

    if not is_valid_text(topic):
        return {
            "summary": "",
            "error": "Health topic is missing."
        }

    if not isinstance(search_results, list) or not search_results:
        return {
            "summary": "",
            "error": "No medical information available for summarization."
        }

    # Keep only valid search results with usable content
    formatted_results = []
    for result in search_results:
        if not isinstance(result, dict):
            continue
        content = result.get("content", "")
        if not is_valid_text(content):
            continue
        formatted_results.append(
            f"Title: {result.get('title', 'N/A')}\n"
            f"URL: {result.get('url', 'N/A')}\n"
            f"Content: {content}"
        )
    if not formatted_results:
        return {
            "summary": "",
            "error": "Search results contain no usable information."
        }

    search_context = "\n\n".join(formatted_results)

    prompt = format_prompt(
        SUMMARY_PROMPT,
        topic=topic,
        search_results=search_context
    )

    try:
        summary = generate_with_gemini(prompt)
        if not is_valid_text(summary):
            return {
                "summary": "",
                "error": "Gemini returned an empty summary."
            }
        return {
            "summary": summary.strip(),
            "error": ""
        }
    except Exception as e:
        return {
            "summary": "",
            "error": f"Summarization failed: {e}"
        }

### - Display Summary Node

In [ ]:
# Display the patient-friendly summary and ask whether to continue to the quiz
def display_summary(state: HealthBotState) -> HealthBotState:
    summary = state.get("summary", "").strip()

    if not summary:
        return {
            "ready_for_quiz": False,
            "error": "Summary is unavailable."
        }

    print("\n" + "=" * 60)
    print("HEALTH INFORMATION")
    print("=" * 60)
    print(summary)
    print("=" * 60)

    while True:
        response = input(
            "\nAre you ready for the comprehension check? (yes/no)\n> "
        ).strip().lower()

        if response in {"yes", "y"}:
            return {
                "ready_for_quiz": True,
                "error": ""
            }

        if response in {"no", "n"}:
            return {
                "ready_for_quiz": False,
                "error": ""
            }

        print("Invalid input. Please enter yes or no.")

In [ ]:
# Route the workflow based on whether the user wants the quiz
def route_quiz(state: HealthBotState) -> str:
    if state.get("ready_for_quiz", False):
        return "generate_quiz"

    return "session_decision"

### - Quiz Generation Node

In [ ]:
# Generate one comprehension question based only on the summary
def generate_quiz(state: HealthBotState) -> HealthBotState:
    summary = state.get("summary", "")

    if not is_valid_text(summary):
        return {
            "quiz_question": "",
            "ready_for_quiz": False,
            "error": "Summary is missing. Cannot generate quiz."
        }
    prompt = format_prompt(
        QUIZ_PROMPT,
        summary=summary
    )
    try:
        question = generate_with_gemini(prompt)
        if not is_valid_text(question):
            return {
                "quiz_question": "",
                "ready_for_quiz": False,
                "error": "Gemini returned an empty quiz question."
            }
        return {
            "quiz_question": question.strip(),
            "ready_for_quiz": True,
            "error": ""
        }
    except Exception as e:
        return {
            "quiz_question": "",
            "ready_for_quiz": False,
            "error": f"Quiz generation failed: {e}"
        }

### - Get Quiz Answer Node


In [ ]:
# Collect the user's answer to the generated quiz question
def get_quiz_answer(state: HealthBotState) -> HealthBotState:
    question = state.get("quiz_question", "")
    
    if not is_valid_text(question):
        return {
            "user_answer": "",
            "error": "Quiz question is missing."
        }

    while True:
        answer = input(
            f"\nQuestion:\n{question}\n\nYour answer:\n> "
        ).strip()

        if not answer:
            print("Please provide an answer before continuing.")
            continue

        return {
            "user_answer": answer,
            "error": ""
        }

### - Grade Answer Node

In [ ]:
# Grade the user's answer using only the generated summary
def grade_answer(state: HealthBotState) -> HealthBotState:
    topic = state.get("topic", "")
    summary = state.get("summary", "")
    quiz_question = state.get("quiz_question", "")
    user_answer = state.get("user_answer", "")

    if not is_valid_text(topic):
        return {
            "grade": "",
            "feedback": "",
            "error": "Health topic is missing."
        }
    if not is_valid_text(summary):
        return {
            "grade": "",
            "feedback": "",
            "error": "Summary is missing."
        }
    if not is_valid_text(quiz_question):
        return {
            "grade": "",
            "feedback": "",
            "error": "Quiz question is missing."
        }
    if not is_valid_text(user_answer):
        return {
            "grade": "",
            "feedback": "",
            "error": "User answer is missing."
        }

    prompt = format_prompt(
        GRADING_PROMPT,
        topic=topic,
        summary=summary,
        quiz_question=quiz_question,
        user_answer=user_answer
    )

    try:
        grading_result = generate_with_gemini(prompt)
        if not is_valid_text(grading_result):
            return {
                "grade": "",
                "feedback": "",
                "error": "Gemini returned an empty grading response."
            }
        grade = extract_grade(grading_result)
        if not grade:
            return {
                "grade": "",
                "feedback": grading_result.strip(),
                "error": "Gemini returned an invalid grade format."
            }
        return {
            "grade": grade,
            "feedback": grading_result.strip(),
            "quiz_completed": True,
            "error": ""
        }
    except Exception as e:
        return {
            "grade": "",
            "feedback": "",
            "quiz_completed": False,
            "error": "..."
        }

### - Display Feedback Node


In [ ]:
# Display the grading result and feedback to the user
def display_feedback(state: HealthBotState) -> HealthBotState:
    grade = state.get("grade", "")
    feedback = state.get("feedback", "")
    error = state.get("error", "")

    if error:
        print(f"\nUnable to evaluate your answer: {error}")
        return state

    if not is_valid_text(feedback):
        print("\nNo feedback was generated.")
        return {
            "error": "Feedback is unavailable."
        }

    print("\n" + "=" * 50)
    print("QUIZ RESULT")
    print("=" * 50)

    if grade:
        print(f"\nGrade: {grade}")

    print("\nFeedback:")
    print(feedback)

    print("=" * 50)

    return state

### - Session Decision Node


In [ ]:
# Ask whether the user wants to learn about another topic
def session_decision(state: HealthBotState) -> HealthBotState:
    while True:
        choice = input(
            "\nWould you like to learn about another health topic? "
            "(yes/no)\n> "
        ).strip().lower()

        if choice in {"yes", "y"}:
            return {
                "continue_session": True,
                "error": ""
            }
        if choice in {"no", "n"}:
            return {
                "continue_session": False,
                "error": ""
            }

        print("Please enter 'yes' or 'no'.")

### - Reset State Node


In [ ]:
# Reset session state before starting a new topic
def reset_state_node(state: HealthBotState) -> HealthBotState:
    return reset_state()

## Langraph Workflow Construction

In [ ]:
# Create the HealthBot state graph
healthbot_graph = StateGraph(HealthBotState)

In [ ]:
# Add workflow nodes to the graph
healthbot_graph.add_node("get_topic", get_topic)
healthbot_graph.add_node("search", search_node)
healthbot_graph.add_node("summarize", summarize_information)
healthbot_graph.add_node("display_summary", display_summary)
healthbot_graph.add_node("generate_quiz", generate_quiz)
healthbot_graph.add_node("get_quiz_answer", get_quiz_answer)
healthbot_graph.add_node("grade_answer", grade_answer)
healthbot_graph.add_node("display_feedback", display_feedback)
healthbot_graph.add_node("session_decision", session_decision)
healthbot_graph.add_node("reset_state", reset_state_node)

In [ ]:
# Connect the main workflow nodes
healthbot_graph.add_edge(START, "get_topic")
healthbot_graph.add_edge("get_topic", "search")
healthbot_graph.add_edge("search", "summarize")
healthbot_graph.add_edge("summarize", "display_summary")

healthbot_graph.add_edge("generate_quiz", "get_quiz_answer")
healthbot_graph.add_edge("get_quiz_answer", "grade_answer")
healthbot_graph.add_edge("grade_answer", "display_feedback")
healthbot_graph.add_edge("display_feedback", "session_decision")

In [ ]:
# Route to the quiz only when the user is ready
healthbot_graph.add_conditional_edges(
    "display_summary",
    route_quiz,
    {
        "generate_quiz": "generate_quiz",
        "session_decision": "session_decision"
    }
)

In [ ]:
# Route the session based on the user's decision
def route_session(state: HealthBotState) -> str:
    if state.get("continue_session", False):
        return "reset_state"

    return END

In [ ]:
# Add conditional routing after the session decision
healthbot_graph.add_conditional_edges(
    "session_decision",
    route_session,
    {
        "reset_state": "reset_state",
        END: END
    }
)

In [ ]:
# Return to topic selection after resetting the session
healthbot_graph.add_edge(
    "reset_state",
    "get_topic"
)

In [ ]:
# Compile the HealthBot workflow
healthbot_app = healthbot_graph.compile()

In [ ]:
# Display the LangGraph workflow and conditional routing
print(healthbot_app.get_graph().draw_mermaid())

## Run the Complete Application

In [ ]:
# Run the complete HealthBot application
initial_state = reset_state()

final_state = healthbot_app.invoke(initial_state)

## Rubric Evidence / Quick Demo

This section provides a quick demonstration of the main HealthBot capabilities and shows how the implementation satisfies the project requirements.

The demo uses the same workflow functions used by the final application.

In [ ]:
# Quick demonstration of the main HealthBot capabilities

print("=" * 70)
print("HEALTHBOT - RUBRIC EVIDENCE")
print("=" * 70)

# 1. API configuration
print("\n[1] API CONFIGURATION")
print("-" * 50)

print(
    "Gemini API key: "
    + ("Loaded" if GEMINI_API_KEY else "Missing")
)

print(
    "Tavily API key: "
    + ("Loaded" if TAVILY_API_KEY else "Missing")
)


# 2. Health-topic validation
print("\n[2] HEALTH TOPIC VALIDATION")
print("-" * 50)

demo_topic = "diabetes"

is_valid, corrected_topic = validate_health_topic(demo_topic)

print(f"Input topic: {demo_topic}")
print(f"Valid health topic: {is_valid}")
print(f"Accepted topic: {corrected_topic}")


if not is_valid:
    raise RuntimeError("Demo topic validation failed.")


# 3. Tavily medical search
print("\n[3] TAVILY MEDICAL SEARCH")
print("-" * 50)

demo_state = {
    "topic": corrected_topic
}

demo_state.update(
    search_node(demo_state)
)

print(
    f"Search results retrieved: "
    f"{len(demo_state.get('search_results', []))}"
)

if demo_state.get("search_results"):
    first_result = demo_state["search_results"][0]

    print(f"First source: {first_result.get('title', 'N/A')}")
    print(f"Source URL: {first_result.get('url', 'N/A')}")


# 4. Gemini summarization
print("\n[4] GEMINI PATIENT-FRIENDLY SUMMARY")
print("-" * 50)

demo_state.update(
    summarize_information(demo_state)
)

summary = demo_state.get("summary", "")

print(summary)

if not summary:
    raise RuntimeError("Demo summary generation failed.")


# 5. Quiz generation
print("\n[5] COMPREHENSION QUIZ")
print("-" * 50)

demo_state["ready_for_quiz"] = True

demo_state.update(
    generate_quiz(demo_state)
)

quiz_question = demo_state.get("quiz_question", "")

print(f"Question: {quiz_question}")

if not quiz_question:
    raise RuntimeError("Demo quiz generation failed.")


# 6. Answer grading
print("\n[6] ANSWER GRADING")
print("-" * 50)

demo_state["user_answer"] = (
    "Diabetes is a condition that affects how the body "
    "regulates blood glucose."
)

demo_state.update(
    grade_answer(demo_state)
)

print(f"Grade: {demo_state.get('grade', 'N/A')}")
print(f"Feedback:\n{demo_state.get('feedback', 'N/A')}")


# 7. LangGraph state
print("\n[7] LANGGRAPH STATE")
print("-" * 50)

print("State fields populated during the workflow:")

for field in [
    "topic",
    "search_results",
    "summary",
    "quiz_question",
    "user_answer",
    "grade",
    "feedback"
]:
    value = demo_state.get(field)

    if isinstance(value, list):
        status = f"{len(value)} result(s)"
    elif is_valid_text(value):
        status = "Populated"
    else:
        status = "Empty"

    print(f"- {field}: {status}")


# 8. Workflow structure
print("\n[8] LANGGRAPH WORKFLOW")
print("-" * 50)

print(healthbot_app.get_graph().draw_mermaid())

print("\n" + "=" * 70)
print("RUBRIC EVIDENCE DEMO COMPLETED")
print("=" * 70)